In [1]:
import zipfile
from collections import Counter

zf = zipfile.ZipFile("../data/asl_citizen/ASL_Citizen.zip")
names = zf.namelist()
print("entries:", len(names))

exts = Counter(n.rsplit(".", 1)[-1].lower() for n in names if not n.endswith("/"))
print("file types:", exts.most_common(10))

tops = Counter(n.split("/")[0] for n in names)
print("top-level:", tops.most_common(10))

for n in names[:15]:
    print("  ", n)
for n in names:
    if not n.lower().endswith((".mp4", ".mov", ".avi", ".webm", "/")):
        print("non-video:", n, zf.getinfo(n).file_size)

entries: 83406
file types: [('mp4', 83399), ('csv', 3), ('txt', 1)]
top-level: [('ASL_Citizen', 83406)]
   ASL_Citizen/
   ASL_Citizen/splits/
   ASL_Citizen/splits/val.csv
   ASL_Citizen/splits/train.csv
   ASL_Citizen/splits/test.csv
   ASL_Citizen/videos/
   ASL_Citizen/videos/5086723486875651-RUN.mp4
   ASL_Citizen/videos/2233415645150214-SWEET.mp4
   ASL_Citizen/videos/09412352698319548-POSSIBLE.mp4
   ASL_Citizen/videos/6184880487426632-PAINT.mp4
   ASL_Citizen/videos/6682045893304822-RACE.mp4
   ASL_Citizen/videos/22425009794174322-SCULPT.mp4
   ASL_Citizen/videos/8563900568328626-SCAR MEMORY.mp4
   ASL_Citizen/videos/6843553981288921-EARTHQUAKE.mp4
   ASL_Citizen/videos/37687322393253075-REPLACE.mp4
non-video: ASL_Citizen/splits/val.csv 510233
non-video: ASL_Citizen/splits/train.csv 1998546
non-video: ASL_Citizen/splits/test.csv 1625987
non-video: ASL_Citizen/use.txt 9925


In [2]:
import pandas as pd

splits = {s: pd.read_csv(zf.open(f"ASL_Citizen/splits/{s}.csv")) for s in ["train", "val", "test"]}
print(splits["train"].columns.tolist())
print(splits["train"].head())
for s, df in splits.items():
    print(s, len(df), "rows")

['Participant ID', 'Video file', 'Gloss', 'ASL-LEX Code']
  Participant ID                        Video file       Gloss ASL-LEX Code
0             P1       15890366051589533-APPLE.mp4       APPLE     A_03_054
1             P1  35618482303951104-IMPOSSIBLE.mp4  IMPOSSIBLE     B_01_032
2             P1         6958143575951994-PARK.mp4        PARK     E_03_028
3             P1     8006032738002744-SOCCER 2.mp4     SOCCER2     F_03_032
4             P1       37542279833186454-STINK.mp4       STINK     H_01_064
train 40154 rows
val 10304 rows
test 32941 rows


In [3]:
print(zf.read("ASL_Citizen/use.txt").decode("utf-8")[:3000])

MICROSOFT RESEARCH LICENSE TERMS
IF YOU LIVE IN THE UNITED STATES, PLEASE READ THE "BINDING ARBITRATION AND CLASS ACTION WAIVER" SECTION BELOW. IT AFFECTS HOW DISPUTES ARE RESOLVED.
These license terms are an agreement between you and Microsoft Corporation (or one of its affiliates). They apply to the source code, object code, or data (collectively "Materials") that accompany this license. IF YOU COMPLY WITH THESE LICENSE TERMS, YOU HAVE THE RIGHTS BELOW. BY USING THE MATERIALS, YOU ACCEPT THESE TERMS.
1) INSTALLATION AND USE RIGHTS TO THE MATERIALS.
Subject to the terms of this agreement, you have the below rights, if applicable, to use the Materials solely for non-commercial, non-revenue generating, research purposes:
a) Source Code. If source code is included, you may use and modify the source code, but you may not distribute the source code. 
b) Object Code. If object code is included, you may use the object code, but you may not distribute the object code.
c) Data. If data is incl

In [4]:
for s, df in splits.items():
    print(s, "| signers:", df["Participant ID"].nunique(), "| glosses:", df["Gloss"].nunique())

train_signers = set(splits["train"]["Participant ID"])
for s in ["val", "test"]:
    overlap = train_signers & set(splits[s]["Participant ID"])
    print(f"{s} signers also in train:", len(overlap))

train_glosses = set(splits["train"]["Gloss"])
print("test glosses missing from train:", len(set(splits["test"]["Gloss"]) - train_glosses))
print("all glosses:", pd.concat(splits.values())["Gloss"].nunique())

train | signers: 35 | glosses: 2731
val | signers: 6 | glosses: 2731
test | signers: 11 | glosses: 2731
val signers also in train: 0
test signers also in train: 0
test glosses missing from train: 0
all glosses: 2731
